# **Replogle dataset preparation**

In [1]:
import scanpy as sc
from spectra.data import data_preprocessing
import pandas as pd
import numpy as np
import anndata as ad
import pertpy as pt


ad.settings.allow_write_nullable_strings = True

In [2]:
adata = pt.data.replogle_2022_rpe1()
adata

/group/sottoriva/michele.calabro/SPECTRA/spectra_venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


AnnData object with n_obs × n_vars = 247914 × 8749
    obs: 'batch', 'gene', 'gene_id', 'transcript', 'gene_transcript', 'guide_id', 'percent_mito', 'UMI_count', 'z_gemgroup_UMI', 'core_scale_factor', 'core_adjusted_UMI_count', 'disease', 'cancer', 'cell_line', 'sex', 'age', 'perturbation', 'organism', 'perturbation_type', 'tissue_type', 'ncounts', 'ngenes', 'nperts', 'percent_ribo', 'celltype'
    var: 'chr', 'start', 'end', 'class', 'strand', 'length', 'in_matrix', 'mean', 'std', 'cv', 'fano', 'ensembl_id', 'ncounts', 'ncells'

In [ ]:
adata = sc.read_h5ad('../data/K562_gwps_raw_singlecell_01.h5ad')
adata

### **Balancing and subsetting the dataset**

In [ ]:
# Separate 'non-targeting' indices
obs = adata.obs[['gene']].copy()
idx_control = obs.index[obs['gene'] == 'non-targeting'] 

# Filter less abundant perturbation conditions
obs_exp = obs[obs['gene'] != 'non-targeting']
counts = obs_exp['gene'].value_counts()
valid_genes = counts[counts >= 50].index

# subset to N cells for most abundand perturbation conditions
idx_experimental = (
    obs_exp[obs_exp['gene'].isin(valid_genes)]
    .groupby('gene', group_keys=False)
    .apply(lambda x: x.sample(n=min(len(x), 500), random_state=42))
    .index
)

# merge with control
keep_indices = idx_control.union(idx_experimental)
adata = adata[adata.obs_names.isin(keep_indices)].copy()

/localscratch/23296.michele.calabro/ipykernel_422075/850199109.py:12: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby('gene', group_keys=False)
/localscratch/23296.michele.calabro/ipykernel_422075/850199109.py:13: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(n=min(len(x), 500), random_state=42))


In [ ]:
adata.var['gene_name'] = adata.var['gene_name'].astype('string')
adata.var_names = adata.var['gene_name']
adata.var_names_make_unique()
adata.var = adata.var.drop(columns=['gene_name'])

# Important: avoid conflict between var index name and var column name
adata.var.index.name = None

In [4]:
adata.var_names_make_unique()
adata.var.index.name = None

### **Reformatting in vcc format style**

In [5]:
# Rename perturbation column (must be 'target_gene') and ctrl samples ('non-targeting')
control_tag = 'non-targeting'
adata.obs = adata.obs.rename(columns={'gene': "target_gene"})
adata.obs["target_gene"] = adata.obs["target_gene"].str.replace(control_tag, "non-targeting", regex=False)
adata.obs['target_gene'] = adata.obs['target_gene'].str.replace(r'\+non-targeting$', '', regex=True)


### **Preprocessing steps**

In [6]:
adata = data_preprocessing(adata, 
    min_genes=200, 
    min_cells=10, 
    min_cells_per_pert=50, 
    logtransform=True
)
adata

View of AnnData object with n_obs × n_vars = 209586 × 8749
    obs: 'batch', 'target_gene', 'gene_id', 'transcript', 'gene_transcript', 'guide_id', 'percent_mito', 'UMI_count', 'z_gemgroup_UMI', 'core_scale_factor', 'core_adjusted_UMI_count', 'disease', 'cancer', 'cell_line', 'sex', 'age', 'perturbation', 'organism', 'perturbation_type', 'tissue_type', 'ncounts', 'ngenes', 'nperts', 'percent_ribo', 'celltype', 'n_genes'
    var: 'chr', 'start', 'end', 'class', 'strand', 'length', 'in_matrix', 'mean', 'std', 'cv', 'fano', 'ensembl_id', 'ncounts', 'ncells', 'n_cells'
    uns: 'log1p'

In [7]:
#NOTE: this preprocessing step retains only perturbations that have an average target expression <30% of control 
# (i.e., >70% knockdown); then, we do the same process but on individual cells, retaining sigle cells with 
# target_gene expression being <50% of the control mean; finally if fewer than 30 cells remain for that perturbation, 
# remove the entire perturbation.

from cell_load.utils.data_utils import filter_on_target_knockdown
adata.var["gene_name"] = adata.var_names.astype(str)
adata_filtered = filter_on_target_knockdown(
    adata=adata,
    perturbation_column="target_gene",       # adata.obs column containing target gene
    control_label="non-targeting",    # control label in that column
    residual_expression=0.40,
    cell_residual_expression=0.60,
    min_cells=30,
    layer=None,                       # use adata.X
    var_gene_name='gene_name',        # gene symbol column in adata.var
    verbose=True,
)
adata_filtered

[input] 209,586 cells | 1611 perturbations (excl. control)
[stage 1 (pert avg filter)] removed 48,759 cells | 355 perturbations
[stage 2 (cell filter)    ] removed 8,106 cells | 0 perturbations
[stage 3 (min cells filter)] removed 0 cells | 0 perturbations
[output] 152,721 cells | 1256 perturbations (excl. control)


AnnData object with n_obs × n_vars = 152721 × 8749
    obs: 'batch', 'target_gene', 'gene_id', 'transcript', 'gene_transcript', 'guide_id', 'percent_mito', 'UMI_count', 'z_gemgroup_UMI', 'core_scale_factor', 'core_adjusted_UMI_count', 'disease', 'cancer', 'cell_line', 'sex', 'age', 'perturbation', 'organism', 'perturbation_type', 'tissue_type', 'ncounts', 'ngenes', 'nperts', 'percent_ribo', 'celltype', 'n_genes'
    var: 'chr', 'start', 'end', 'class', 'strand', 'length', 'in_matrix', 'mean', 'std', 'cv', 'fano', 'ensembl_id', 'ncounts', 'ncells', 'n_cells', 'gene_name'
    uns: 'log1p'

### **Saving**

In [9]:
adata.write('../../data/rep_rpe1/replogle_rpe1_standard_filtered.h5ad')